# 🎮 EchoForest AI - Baseline Test (Step 0)

**목적**: 기존 unSmile 모델로 게임 키워드 데이터 테스트

**환경**: NVIDIA L40S GPU, Python 3.12, PyTorch 2.5.1+cu121

## 1. 환경 설정 및 라이브러리 로드

In [ ]:
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support
import warnings
warnings.filterwarnings('ignore')

# 설정
MODEL_NAME = "smilegate-ai/kor_unsmile"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
THRESHOLD = 0.5

LABEL_NAMES = [
    "여성/가족", "남성", "성소수자", "인종/국적", "연령",
    "지역", "종교", "기타 혐오", "악플/욕설", "clean"
]

print("=" * 60)
print("EchoForest AI - Baseline Test")
print("=" * 60)
print(f"Device: {DEVICE}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print("=" * 60)

## 2. 모델 로드

In [ ]:
print("모델 로딩 중...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model = model.to(DEVICE)
model.eval()
print("✓ 모델 로드 완료!")

## 3. 테스트 데이터 로드

In [ ]:
# 경로 설정 (환경에 맞게 수정하세요)
DATA_PATH = "../_archive/2_Baseline_Test/keywords_unsmile_format.tsv"  # _archive로 이동
# DATA_PATH = "/content/keywords_unsmile_format.tsv"  # Colab

df = pd.read_csv(DATA_PATH, sep='\t', encoding='utf-8')
print(f"✓ 데이터 로드 완료: {len(df)}개 문장")
print(f"  - Clean: {df['clean'].sum()}개")
print(f"  - 악플/욕설: {df['악플/욕설'].sum()}개")

df.head(10)

## 4. 예측 함수 정의

In [ ]:
def predict_batch(texts, batch_size=32):
    """배치 예측"""
    all_probs = []
    
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]
        inputs = tokenizer(
            batch_texts.tolist(), 
            return_tensors="pt", 
            truncation=True, 
            max_length=128, 
            padding=True
        )
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
        
        with torch.no_grad():
            outputs = model(**inputs)
            probs = torch.sigmoid(outputs.logits).cpu().numpy()
        
        all_probs.extend(probs)
    
    return np.array(all_probs)

## 5. 예측 실행

In [ ]:
print("예측 실행 중...")
probs = predict_batch(df['문장'].values)
predictions = (probs > THRESHOLD).astype(int)

# 악플/욕설 (index 8)과 clean (index 9) 추출
abuse_probs = probs[:, 8]
clean_probs = probs[:, 9]
abuse_preds = predictions[:, 8]
clean_preds = predictions[:, 9]

# 실제 라벨
y_true_abuse = df['악플/욕설'].values
y_true_clean = df['clean'].values

print("✓ 예측 완료!")

## 6. 📊 악플/욕설 분류 결과

In [ ]:
print("📊 악플/욕설 분류 결과")
print("=" * 60)
print(classification_report(y_true_abuse, abuse_preds, target_names=['Non-Abuse', 'Abuse']))

## 7. 📊 Clean 분류 결과

In [ ]:
print("📊 Clean 분류 결과")
print("=" * 60)
print(classification_report(y_true_clean, clean_preds, target_names=['Non-Clean', 'Clean']))

## 8. 📈 핵심 성능 지표 요약

In [ ]:
abuse_precision, abuse_recall, abuse_f1, _ = precision_recall_fscore_support(
    y_true_abuse, abuse_preds, average='binary'
)
clean_precision, clean_recall, clean_f1, _ = precision_recall_fscore_support(
    y_true_clean, clean_preds, average='binary'
)

print("📈 핵심 성능 지표 요약")
print("=" * 60)
print(f"악플/욕설 - Precision: {abuse_precision:.4f}, Recall: {abuse_recall:.4f}, F1: {abuse_f1:.4f}")
print(f"Clean     - Precision: {clean_precision:.4f}, Recall: {clean_recall:.4f}, F1: {clean_f1:.4f}")

## 9. ❌ 인식 실패 케이스 분석

In [ ]:
# 악플/욕설인데 clean으로 예측된 경우 (False Negative - 가장 중요!)
fn_mask = (y_true_abuse == 1) & (abuse_preds == 0)
false_negatives = df[fn_mask]['문장'].values

print(f"🔴 악플/욕설 → Clean으로 오분류 (False Negative): {len(false_negatives)}건")
print("   ↳ Fine-tuning 시 우선 개선 필요!")
print()

if len(false_negatives) > 0:
    fn_df = df[fn_mask][['문장']].copy()
    fn_df['욕설확률'] = abuse_probs[fn_mask]
    fn_df = fn_df.sort_values('욕설확률', ascending=False)
    display(fn_df.head(30))

In [ ]:
# Clean인데 악플/욕설로 예측된 경우 (False Positive)
fp_mask = (y_true_clean == 1) & (abuse_preds == 1)
false_positives = df[fp_mask]['문장'].values

print(f"🟡 Clean → 악플/욕설로 오분류 (False Positive): {len(false_positives)}건")
print("   ↳ 게임 컨텍스트 오탐 가능성")
print()

if len(false_positives) > 0:
    fp_df = df[fp_mask][['문장']].copy()
    fp_df['욕설확률'] = abuse_probs[fp_mask]
    fp_df = fp_df.sort_values('욕설확률', ascending=False)
    display(fp_df.head(30))

## 10. 📊 시각화

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# 1. 악플/욕설 Confusion Matrix
ax1 = axes[0, 0]
cm_abuse = confusion_matrix(y_true_abuse, abuse_preds)
sns.heatmap(cm_abuse, annot=True, fmt='d', cmap='Blues', ax=ax1,
            xticklabels=['Pred: Non-Abuse', 'Pred: Abuse'],
            yticklabels=['True: Non-Abuse', 'True: Abuse'])
ax1.set_title('Abuse Classification - Confusion Matrix', fontsize=12, fontweight='bold')

# 2. Clean Confusion Matrix
ax2 = axes[0, 1]
cm_clean = confusion_matrix(y_true_clean, clean_preds)
sns.heatmap(cm_clean, annot=True, fmt='d', cmap='Greens', ax=ax2,
            xticklabels=['Pred: Non-Clean', 'Pred: Clean'],
            yticklabels=['True: Non-Clean', 'True: Clean'])
ax2.set_title('Clean Classification - Confusion Matrix', fontsize=12, fontweight='bold')

# 3. 확률 분포
ax3 = axes[1, 0]
abuse_data = pd.DataFrame({
    'Probability': abuse_probs,
    'Actual': ['Negative' if x == 1 else 'Clean' for x in y_true_abuse]
})
colors = {'Negative': 'red', 'Clean': 'green'}
for label, color in colors.items():
    subset = abuse_data[abuse_data['Actual'] == label]
    ax3.hist(subset['Probability'], bins=20, alpha=0.6, label=label, color=color)
ax3.axvline(x=0.5, color='black', linestyle='--', label='Threshold (0.5)')
ax3.set_xlabel('Abuse Probability')
ax3.set_ylabel('Count')
ax3.set_title('Abuse Probability Distribution', fontsize=12, fontweight='bold')
ax3.legend()

# 4. 성능 지표 바 차트
ax4 = axes[1, 1]
metrics = ['Precision', 'Recall', 'F1-Score']
abuse_scores = [abuse_precision, abuse_recall, abuse_f1]
clean_scores = [clean_precision, clean_recall, clean_f1]

x = np.arange(len(metrics))
width = 0.35

bars1 = ax4.bar(x - width/2, abuse_scores, width, label='Abuse', color='red', alpha=0.7)
bars2 = ax4.bar(x + width/2, clean_scores, width, label='Clean', color='green', alpha=0.7)

ax4.set_ylabel('Score')
ax4.set_title('Performance Metrics', fontsize=12, fontweight='bold')
ax4.set_xticks(x)
ax4.set_xticklabels(metrics)
ax4.set_ylim(0, 1.1)
ax4.legend()

for bar in bars1 + bars2:
    height = bar.get_height()
    ax4.annotate(f'{height:.3f}', xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('baseline_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. 결과 저장

In [ ]:
# 결과 DataFrame
results_df = df.copy()
results_df['pred_abuse'] = abuse_preds
results_df['pred_clean'] = clean_preds
results_df['prob_abuse'] = abuse_probs
results_df['prob_clean'] = clean_probs
results_df['correct_abuse'] = y_true_abuse == abuse_preds
results_df['correct_clean'] = y_true_clean == clean_preds

# CSV로 저장
results_df.to_csv('baseline_test_results.csv', index=False, encoding='utf-8-sig')
print("✓ 결과 저장: baseline_test_results.csv")

## 12. 🎯 최종 요약

In [ ]:
print("=" * 60)
print("🎯 Baseline 테스트 최종 요약")
print("=" * 60)
print(f"""
총 테스트 문장: {len(df)}개
├── Clean: {df['clean'].sum()}개
└── 악플/욕설: {df['악플/욕설'].sum()}개

📊 악플/욕설 탐지 성능:
├── Precision: {abuse_precision:.4f}
├── Recall: {abuse_recall:.4f} ← 🎯 핵심 지표!
└── F1-Score: {abuse_f1:.4f}

📊 Clean 탐지 성능:
├── Precision: {clean_precision:.4f}
├── Recall: {clean_recall:.4f}
└── F1-Score: {clean_f1:.4f}

❌ 인식 실패 케이스:
├── False Negative (욕설 → Clean): {len(false_negatives)}건
└── False Positive (Clean → 욕설): {len(false_positives)}건

💡 Fine-tuning 개선 목표:
├── 현재 악플/욕설 Recall: {abuse_recall:.4f}
└── 목표 악플/욕설 Recall: ≥ 0.75
""")
print("=" * 60)
print("✅ Baseline 테스트 완료!")